# piv_compare.ipynb
**Does:** trains the shared hybrid_segment filter (Part 0), then compares every bulk-motion method with every tracker on the 13 µm/px sequences, plus a large-motion stress test.
**Uses:** `src/scale.py`, `src/hybrid_segment.py`, `src/piv_compare.py` (and the `run_13um` data made by pipeline.ipynb).
**Outputs:** `hybrid_segment_outputs/` (filter, cached detections) and `piv_outputs/<n>/` (motion and tracking tables, stress test, figures).

## Settings

In [ ]:
# settings for this run
DATA_NAME  = 'run_13um'
RUN_ID     = None              # None = new output folder; a number = continue that run
SPLITS     = ('val', 'test')
DETECTIONS = 'hybrid_segment'  # 'hybrid_segment', 'song' (Song's double threshold) or 'truth' (perfect detections)
QF_FILE    = 'hybrid_segment_outputs/quality_filter.joblib'   # trained in Part 0 if missing
QF_TAU     = 0.3
QF_TRAIN_STEP = 2
MOTIONS    = None              # None = all (raft only on a GPU)
TRACKERS   = None              # None = all
MAX_FRAMES = None              # a number = only the first n frames (quick test)
FIG_MOTION, FIG_TRACKER = 'song_piv', 'song'   # combination drawn in Part D
STRESS_PAIRS, STRESS_MOVE = 4, 6               # stress test: frames used, particles moved per frame
STRESS_MOTIONS = None          # None = same as MOTIONS

## Setup

In [ ]:
# mount Drive, load the scripts, read the pixel scale from the data and create the output folder
import os, sys, glob, json, shutil
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive
from IPython.display import display

drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/snocam_pipeline'
DATA = f'/content/{DATA_NAME}'
sh = get_ipython().system
sh('pip install -q openpiv')
shutil.copytree(f'{DRIVE}/src', '/content/src', dirs_exist_ok=True, ignore=shutil.ignore_patterns('__pycache__'))
sys.path.append('/content/src')


def restore(name):
    # copy a dataset from Drive to /content (from its .tar if there is one)
    if not os.path.exists(f'/content/{name}'):
        if os.path.exists(f'{DRIVE}/{name}.tar'):
            sh(f'tar -xf "{DRIVE}/{name}.tar" -C /content')
        elif os.path.exists(f'{DRIVE}/{name}'):
            shutil.copytree(f'{DRIVE}/{name}', f'/content/{name}')
    return os.path.exists(f'/content/{name}')


assert restore(DATA_NAME), f'no {DATA_NAME} in {DRIVE}: run pipeline.ipynb Part A first'
import scale as SC
import piv_compare as pc
res = SC.configure_from_data(DATA)
print('resolution:', res)
MOTIONS = MOTIONS or pc.default_motions()
TRACKERS = TRACKERS or list(pc.LINKERS)

os.makedirs(f'{DRIVE}/piv_outputs', exist_ok=True)
if RUN_ID is None:
    used = [int(n) for n in os.listdir(f'{DRIVE}/piv_outputs') if n.isdigit()]
    RUN_ID = max(used) + 1 if used else 0
OUT = f'{DRIVE}/piv_outputs/{RUN_ID}'
os.makedirs(OUT, exist_ok=True)
json.dump(dict(DATA_NAME=DATA_NAME, resolution=res, SPLITS=list(SPLITS), DETECTIONS=DETECTIONS, QF_TAU=QF_TAU,
               MOTIONS=MOTIONS, TRACKERS=TRACKERS, pixel_settings=pc.settings()),
          open(f'{OUT}/settings.json', 'w'), indent=1, default=str)
print('outputs:', OUT, '\nmotion methods:', MOTIONS, '\ntrackers:', TRACKERS)

## Status

In [ ]:
# what is ready and what is already done
seqs = [d for d in sorted(glob.glob(f'{DATA}/seq*')) if d.split('_')[-1] in SPLITS]
ready = [d for d in seqs if glob.glob(f'{d}/bgrem/bgrem_*.png') and glob.glob(f'{d}/truth/labels/frame_*.png')]
print(f'{len(ready)}/{len(seqs)} sequences ready | filter trained:', os.path.exists(f'{DRIVE}/{QF_FILE}'),
      '| sequences scored:', len(glob.glob(f'{OUT}/per_seq/*_motion.csv')))

## Part 0: hybrid_segment filter

In [ ]:
# load the shared quality filter, or train it on the training sequences the first time
if DETECTIONS == 'hybrid_segment':
    pc.use_hybrid_segment(f'{DRIVE}/{QF_FILE}', QF_TAU, data_dir=DATA, frame_step=QF_TRAIN_STEP)
    curve = os.path.join(os.path.dirname(f'{DRIVE}/{QF_FILE}'), 'precision_curve.csv')
    if os.path.exists(curve):
        display(pd.read_csv(curve).set_index('tau').T)

## Part A: every motion method x every tracker

In [ ]:
# run all combinations on every sequence (resumable per sequence)
motion_table, tracking_table = pc.run(DATA, OUT, SPLITS, MOTIONS, TRACKERS, DETECTIONS, MAX_FRAMES)

## Part B: results

In [ ]:
# motion accuracy and tracking scores (%)
pd.set_option('display.max_columns', None, 'display.width', 250)
motion_table, tracking_table = pc.results(OUT)
display(motion_table.sort_values('Median error px'))
display(tracking_table.sort_values('Counted arrivals correct %', ascending=False))

In [ ]:
# the key comparison: links found for particles that moved a lot on their own, and arrivals counted correctly
col = pc.own_column()
display(tracking_table[col].unstack('motion'))
display(tracking_table['Counted arrivals correct %'].unstack('motion'))
C = pd.concat([pd.read_csv(f) for f in sorted(glob.glob(f'{OUT}/per_seq/*_tracking.csv'))])
per_seq = pc.summarise_tracking(C[C.tracker.isin(['song', 'tracking']) & (C.motion == 'ransac')], by=('seq_id', 'tracker', 'motion'))
display(per_seq.droplevel('motion')[[col, 'Counted arrivals correct %', 'Counted arrivals, % of true', 'New arrivals found %',
                                     'Particles never broken %']])

## Part C: motion fields

In [ ]:
# true displacement vs each method's prediction on the frame pair with the most motion
SEQ = ready[-1]
S = pc.load_seq(SEQ, 'truth', MAX_FRAMES)
pairs = pc.truth_pairs(pc.truth_table(S['truth']))
PAIR = int(pairs[pairs.usable].groupby('frame').disp.quantile(0.9).idxmax())
del S
pc.field_figure(SEQ, PAIR, MOTIONS, f'{OUT}/field_{os.path.basename(SEQ)}_{PAIR:04d}.png'); plt.show()

## Part D: trajectories

In [ ]:
# Song-style trajectories of the particles that moved most on their own
for m, t in [(FIG_MOTION, FIG_TRACKER), (FIG_MOTION, 'reid')]:
    if t in TRACKERS and m in MOTIONS:
        pc.seq_figures(SEQ, OUT, m, t, n=4, detections=DETECTIONS, max_frames=MAX_FRAMES)
        plt.show()

## Part E: large-motion stress test

In [ ]:
# copy a frame, move it by a known bulk motion and move a few particles further on their own (resumable)
stress_table, R = pc.run_stress(DATA, OUT, SPLITS, STRESS_MOTIONS or MOTIONS, TRACKERS, STRESS_PAIRS, pc.STRESS_BULK,
                                pc.STRESS_DISP, STRESS_MOVE)
display(stress_table)
pc.stress_curves(R, f'{OUT}/stress_curves.png'); plt.show()

In [ ]:
# one moved particle as each combination sees it
seq_dir, frame = pc.stress_frames(DATA, SPLITS, STRESS_PAIRS)[0]
combos = [(m, t) for m, t in [('song_piv', 'song'), ('song_piv', 'tracking'), ('song_piv', 'hungarian'),
                              ('song_piv', 'reid'), ('tvl1', 'nearest'), ('raft', 'nearest')]
          if m in (STRESS_MOTIONS or MOTIONS) and t in TRACKERS]
DISP = pc.STRESS_DISP[3]
pc.stress_example(seq_dir, frame, pc.STRESS_BULK[-1], DISP, combos, f'{OUT}/stress_example_{DISP}px.png',
                  n_move=STRESS_MOVE, seed=0); plt.show()